In [38]:
import pandas as pd

In [39]:
df = pd.read_excel("processing_data_4.xlsx")

In [40]:
# ===== REMOVE OUTLIERS =====
def remove_outliers_iqr(df, col):
    if col not in df.columns:
        return df
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[col] >= lower) & (df[col] <= upper)]

# Áp dụng cho các cột chính
for col in ["price", "quantity_sold"]:
    if col in df.columns:
        df = remove_outliers_iqr(df, col)

# Loại nhiễu theo logic nghiệp vụ
if "rating_average" in df.columns:
    df = df[(df["rating_average"] >= 1) & (df["rating_average"] <= 5)]

if "discount_percent" in df.columns:
    df = df[(df["discount_percent"] >= 0) & (df["discount_percent"] <= 100)]

# Reset index sau khi lọc
df = df.reset_index(drop=True)

In [41]:
# ===== SELECT & REORDER COLUMNS =====
col_order = []

# 1. Identifiers
col_order += [c for c in ["id", "cluster_id", "dup_component_id"] if c in df.columns]

# 2. Original features
col_order += [c for c in ["price", "discount_percent", "rating_average", "quantity_sold", "seller_id", "brand"] if c in df.columns]

# 3. Labels (Supervised)
col_order += [c for c in df.columns if c.startswith("is_duplicate") or c.startswith("pred_") or c.startswith("proba_")]

# 4. Cluster profiles
col_order += [c for c in ["avg_price", "avg_rating", "avg_quantity_sold", "dup_ratio"] if c in df.columns]

# 5. Business tags
col_order += [c for c in ["price_segment", "rating_tag", "popularity", "cluster_meaning"] if c in df.columns]

# 6. Graph features
col_order += [c for c in ["dup_degree"] if c in df.columns]

# 7. PCA components
col_order += [c for c in df.columns if c.startswith("pca")]

# Thêm các cột còn lại
remaining_cols = [c for c in df.columns if c not in col_order]
col_order += remaining_cols

In [42]:
df_clean = df[col_order]

In [43]:
df_clean.to_excel("processing_data_5.xlsx", index=False)
print("[INFO] Đã lưu processing_data_5.xlsx")

[INFO] Đã lưu processing_data_5.xlsx
